# 10.6 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/10-deployment-mlops/06-monitoring-drift/solutions.ipynb)

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
repo_file("data/download.py")                                  # churn_model finds the data through it
sys.path.insert(0, str(repo_file("lib/churn_model.py").parent))
import churn_model as cm

bundle = cm.load_bundle()          # trains and saves models/churn_bundle.joblib the first time
ROOT = cm.repo_root()
print(f"Model trained {bundle['trained']}, test AUC {bundle['metrics']['test_auc']}, "
      f"threshold {bundle['threshold']}")

Model trained 2026-10-28, test AUC 0.741, threshold 0.07


## Exercise 1

In [2]:
e = np.array([0.30, 0.30, 0.20, 0.10, 0.10])
a = np.array([0.20, 0.25, 0.25, 0.15, 0.15])
terms = (a - e) * np.log(a / e)
print(terms.round(4), f"PSI = {terms.sum():.4f}")

[0.0405 0.0091 0.0112 0.0203 0.0203] PSI = 0.1014


Terms: 0.0405, 0.0091, 0.0112, 0.0203, 0.0203 → about **0.101**, right on the border between stable and watch. Small shifts in several bins add up.

In [3]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

sales = cm.load_sales(both_years=True)         # Dec 2009 - Dec 2011
HISTORY_DAYS = 274                              # same window as the production model (Dec 2010 - Aug 2011)
CUTOFFS = pd.date_range("2010-09-01", "2011-09-01", freq="MS")
tables = {c: cm.build_customer_table(sales, c, lookback_days=HISTORY_DAYS) for c in CUTOFFS}
print(f"{len(tables)} monthly customer tables, {CUTOFFS[0].date()} to {CUTOFFS[-1].date()}")

def new_model():
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

13 monthly customer tables, 2010-09-01 to 2011-09-01


## Exercise 2

In [4]:
train = tables[CUTOFFS[0]]
model = new_model().fit(train[cm.FEATURES], train["churned"])

def calibration_alert(predicted, actual, tolerance=0.10):
    return abs(actual - predicted) > tolerance

for cutoff in CUTOFFS[1:]:
    t = tables[cutoff]
    pred, act = model.predict_proba(t[cm.FEATURES])[:, 1].mean(), t["churned"].mean()
    known = (cutoff + pd.Timedelta(days=91)).strftime("%Y-%m-%d")
    flag = "ALERT" if calibration_alert(pred, act) else ""
    print(f"{cutoff:%Y-%m}  predicted {pred:.3f}  actual {act:.3f}  known {known}  {flag}")

2010-10  predicted 0.411  actual 0.459  known 2010-12-31  
2010-11  predicted 0.401  actual 0.545  known 2011-01-31  ALERT
2010-12  predicted 0.391  actual 0.648  known 2011-03-02  ALERT
2011-01  predicted 0.403  actual 0.650  known 2011-04-02  ALERT
2011-02  predicted 0.413  actual 0.637  known 2011-05-03  ALERT
2011-03  predicted 0.423  actual 0.611  known 2011-05-31  ALERT
2011-04  predicted 0.430  actual 0.596  known 2011-07-01  ALERT
2011-05  predicted 0.435  actual 0.589  known 2011-07-31  ALERT
2011-06  predicted 0.431  actual 0.610  known 2011-08-31  ALERT
2011-07  predicted 0.429  actual 0.580  known 2011-09-30  ALERT
2011-08  predicted 0.425  actual 0.520  known 2011-10-31  
2011-09  predicted 0.424  actual 0.436  known 2011-12-01  


It fires from November 2010 to July 2011, but each alert is only known three months later: the December problem is visible at the start of March. That delay is why teams also watch inputs and scores (available at once) and, where possible, early proxies for the label (for example "no visit to the website for 30 days").

## Exercise 3

In [5]:
rows = []
for cutoff in pd.date_range("2011-06-01", "2011-09-01", freq="MS"):
    newest = cutoff - pd.DateOffset(months=3)              # its 91-day labels are (just) complete
    rolling = new_model().fit(tables[newest][cm.FEATURES], tables[newest]["churned"])
    t = tables[cutoff]
    for name, m in [("fixed Sep 2010", model), (f"rolling ({newest:%b %Y})", rolling)]:
        p = m.predict_proba(t[cm.FEATURES])[:, 1]
        rows.append({"month": f"{cutoff:%Y-%m}", "model": name, "AUC": roc_auc_score(t["churned"], p),
                     "gap": t["churned"].mean() - p.mean()})
pd.DataFrame(rows).round(3)

,month,model,AUC,gap
0,2011-06,fixed Sep 2010,0.778,0.178
1,2011-06,rolling (Mar 2011),0.780,-0.005
2,2011-07,fixed Sep 2010,0.778,0.151
3,2011-07,rolling (Apr 2011),0.783,-0.012
4,2011-08,fixed Sep 2010,0.774,0.095
5,2011-08,rolling (May 2011),0.775,-0.053
6,2011-09,fixed Sep 2010,0.742,0.012
7,2011-09,rolling (Jun 2011),0.740,-0.157


AUC is about the same for both policies. The calibration gap decides it: the rolling model is better in early summer, when the newest data happens to be similar, and worse by September, when it carries over the spring's higher churn. Neither "always retrain" nor "never retrain" wins every month; a champion/challenger comparison on the newest labelled month (10.5) does.